# 한국 배당성장주 알파 팩터 연구실 · 과적합 차단 전진검증 (Google Colab)

**목표:** 현행 배당성장 규칙(전략 A: `1~8번` 품질 필터)을 통과한 종목군 안에서, **과적합(Overfitting)을 차단하면서 코스피(배당 포함 TR 추정치 포함)를 안정적으로 이기는 선별·비중 규칙**을 과학적으로 검증합니다.

### 과적합 차단 4중 검증 프레임워크
1. **단일 팩터 예측력(IC · 5분위) 검증**: 경제적 근거가 명확한 **7개 핵심 팩터**(밸류, 배당성장, 수익성 스프레드, 총주주환원, 12-1M 모멘텀, 200일선 추세, 저변동성)가 실제로 다음 1년 초과수익을 예측했는지 연도별 검증.
2. **Walk-Forward(전진 검증, 100% Out-of-Sample) 학습**: 매해 **직전 연도까지의 과거 데이터만**으로 팩터 가중치를 정규화(Shrinkage) 학습하고, **한 번도 보지 않은 다음 해** 실제 일별 시세로 성과를 측정.
3. **3종 스트레스 테스트**: ① 최고 대박 1종목 제거(Leave-One-Out) 검증 ② 원칙 부합 종목 무작위 추출 500회 몬테카를로(`p-value`) ③ 보유 종목 수(`TOP_N = 8, 10, 12, 15`) 민감도.
4. **엄격한 실전 채택 관문(Go / Hold / Reject)**: 4개 관문을 모두 통과한 규칙만 `colab_screener.ipynb` 반영 후보로 추천하고, 아니면 *"과적합 우려 — 현행 전략 A 유지"*라고 솔직히 판정합니다.

### 중간에 끊겨도 이어서 실행하는 방법 (Google Drive 체크포인트)
- **일별 주가 수집(40종목마다)** · **연도별 팩터 패널** · **몬테카를로 500회 시뮬레이션(50회마다)**이 모두 Google Drive(`Colab Notebooks/dividend_alpha_lab/`)에 자동 저장됩니다.
- 모바일 화면 꺼짐이나 Colab 세션 종료로 중간에 끊겨도, **다시 열어서 `런타임 → 모두 실행`만 누르면 완료된 단계를 0초 만에 건너뛰고 끊긴 지점부터 바로 이어서 실행**됩니다!
- **Colab Pro 백그라운드 실행**: 우측 상단 `런타임 유형 변경`에서 백그라운드 실행을 켜 두시면 브라우저를 닫아도 끝까지 완료되어 Drive에 PDF가 저장됩니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정, 한글 폰트(fonts-nanum) 및 PDF 도구 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, re, io, gzip, json, math, time, base64, random, warnings, statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "finance-datareader"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")


In [ ]:
# ==============================================================================
# 2. 실험 설정 · Google Drive 연결 · 끊김 이어받기(Checkpoint) 엔진
# ==============================================================================
LAB_CONFIG = {
    # 검증 기간 (FY2018 -> 2019-04 출발 ~ 최신, 2020 코로나 폭락장 포함 ~7.7년)
    "START_FY": 2018,
    "END_FY": 2024,
    "TOP_N": 10,
    "INITIAL_CAPITAL": 100_000_000,
    "FEE_RATE": 0.0025,            # 편도 거래비용 0.25%
    "DIV_TAX_RATE": 0.154,         # 배당소득세 15.4%
    "KOSPI_EST_DIV_YIELD": 0.018,  # KOSPI 배당 포함(TR) 추정용 연 세후 배당수익률 (+1.8%/년)

    # 과적합 차단 설정
    "MIN_LIQ_EOK": 10.0,           # 60일 일평균 거래대금 10억 원 이상 (이전 Ablation 검증 반영)
    "VAL_CAP": 0.50,               # 밸류 괴리율 ±50% 윈저라이징
    "WF_SHRINKAGE": 0.50,          # Walk-Forward 가중치 수축 계수 (0.5 = 학습 50% + 균등 50%, 과적합 방지!)
    "WF_MAX_WEIGHT": 0.40,         # 단일 팩터 최대 가중치 상한 40%
    "MC_RUNS": 500,                # 몬테카를로 무작위 포트폴리오 시뮬레이션 횟수 (체크포인트 이어받기 지원)
    "MC_SEED": 42,

    # Google Drive 체크포인트 & 리포트 저장 (끊겨도 자동 이어받기!)
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_alpha_lab",
    "FORCE_REFRESH_PRICES": False, # True: 시세 캐시 무시하고 새로 수집
    "FORCE_REFRESH_MC": False,     # True: 몬테카를로 500회 처음부터 다시 계산

    # 공개 데이터 소스 (Common_Everything Public)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
    "CACHE_DIR": "./_cache_alpha_lab",
    "WORKERS": 8,
}

CACHE_DIR = Path(LAB_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
if IN_COLAB and LAB_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(LAB_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 체크포인트·리포트 경로: {DRIVE_ROOT}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 임시 저장소로 진행합니다 ({type(e).__name__}: {e})")

def save_ckpt_json(filename: str, obj: dict) -> None:
    raw = gzip.compress(json.dumps(obj, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    (CACHE_DIR / filename).write_bytes(raw)
    if DRIVE_ROOT is not None:
        try:
            tmp = DRIVE_ROOT / (filename + ".tmp")
            dst = DRIVE_ROOT / filename
            tmp.write_bytes(raw)
            tmp.replace(dst)
        except Exception as e:
            print(f"  [주의] Drive 체크포인트 저장 지연({type(e).__name__})")

def load_ckpt_json(filename: str, force: bool = False) -> dict:
    if force:
        return {}
    for base in ([DRIVE_ROOT, CACHE_DIR] if DRIVE_ROOT is not None else [CACHE_DIR]):
        p = base / filename
        if p.exists() and p.stat().st_size > 0:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                continue
    return {}

print(f"[설정] FY{LAB_CONFIG['START_FY']}~FY{LAB_CONFIG['END_FY']} (2019-04~최신) | TOP_N={LAB_CONFIG['TOP_N']} | "
      f"Walk-Forward 수축={LAB_CONFIG['WF_SHRINKAGE']:.0%} | 몬테카를로 {LAB_CONFIG['MC_RUNS']}회")


In [ ]:
# ==============================================================================
# 3. 공개 PIT 재무·배당 데이터 로드 + 일별 주가 이어받기 수집 (40종목마다 Drive 저장)
# ==============================================================================
def load_public_data_bytes(filename: str) -> bytes:
    for p in [Path("../data") / filename, Path("dividend-backtest/data") / filename, CACHE_DIR / filename]:
        if p.exists():
            return p.read_bytes()
    for br in LAB_CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{LAB_CONFIG['GITHUB_REPO']}/{br}/{LAB_CONFIG['DATA_SUBPATH']}/{filename}"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 100:
                (CACHE_DIR / filename).write_bytes(r.content)
                return r.content
        except Exception:
            continue
    raise RuntimeError(f"공개 데이터({filename})를 찾을 수 없습니다.")

pit_raw = gzip.decompress(load_public_data_bytes("kr_fundamentals_pit.jsonl.gz")).decode("utf-8")
COMPANIES = [json.loads(line) for line in pit_raw.splitlines() if line.strip()]
rf_df = pd.read_csv(io.StringIO(load_public_data_bytes("kr_treasury_10y_annual.csv").decode("utf-8")))
RF_MAP = {int(r["연도"]): float(r["국고채10년_연평균_퍼센트"]) / 100.0 for _, r in rf_df.iterrows()}
print(f"[데이터 로드] 공개 PIT 기업 {len(COMPANIES)}개사 | 국고채 10년물 {min(RF_MAP)}~{max(RF_MAP)}년")

_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-alpha-lab)"}

def fetch_naver_ohlcv(symbol: str, count: int = 2600) -> list[list]:
    """네이버 fchart 일별 수정주가·거래량 수집 -> [[YYYYMMDD, close, volume], ...]"""
    for attempt in range(3):
        try:
            r = requests.get(_FCHART_URL, params={"symbol": symbol, "timeframe": "day", "count": count, "requestType": 0}, headers=_HEADERS, timeout=20)
            r.raise_for_status()
            out = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                parts = m.group(1).split("|")
                c = float(parts[4])
                v = float(parts[5])
                if c > 0 and parts[0] >= "20170101":
                    out.append([parts[0], c, v])
            if out:
                return out
        except Exception:
            time.sleep(0.35 * (attempt + 1))
    return []

KST = timezone(timedelta(hours=9))
today_tag = datetime.now(KST).strftime("%Y-%m-%d")
CKPT_PX_NAME = f"ckpt_prices_daily_{today_tag[:7]}.json.gz"   # 월 단위 시세 캐시 재사용 (같은 달 재실행 시 1초!)
px_ckpt = load_ckpt_json(CKPT_PX_NAME, force=LAB_CONFIG["FORCE_REFRESH_PRICES"])
px_store = px_ckpt.get("data", {})

all_symbols = ["KOSPI"] + [c["ticker"] for c in COMPANIES]
todo_syms = [s for s in all_symbols if s not in px_store]

if px_store and not todo_syms:
    print(f"[시세 이어받기 완료] Drive 캐시에서 {len(px_store)}개 종목 일별 주가 즉시 복원!")
else:
    print(f"[시세 수집] 보유 캐시 {len(px_store)}개 -> 남은 {len(todo_syms)}개 수집 시작 (40종목마다 Drive 자동 저장)...")
    t0 = time.time()
    try:
        with ThreadPoolExecutor(max_workers=LAB_CONFIG["WORKERS"]) as ex:
            futs = {ex.submit(fetch_naver_ohlcv, s): s for s in todo_syms}
            for idx, fut in enumerate(as_completed(futs), 1):
                sym = futs[fut]
                rows = fut.result()
                if rows:
                    px_store[sym] = rows
                if idx % 40 == 0 or idx == len(futs):
                    save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})
                    print(f"  시세 진행 {len(px_store)}/{len(all_symbols)} ({time.time()-t0:.0f}초, Drive 저장됨)")
    finally:
        if todo_syms:
            save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})

def _to_df(rows: list[list]) -> pd.DataFrame:
    df = pd.DataFrame(rows, columns=["date", "close", "volume"])
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    return df.set_index("date").sort_index()

KOSPI_DF = _to_df(px_store["KOSPI"])
PRICE_MAP = {t: _to_df(rows) for t, rows in px_store.items() if t != "KOSPI" and rows}
CALENDAR = KOSPI_DF.index
print(f"[완료] 일별 수정주가 준비 완료: {len(PRICE_MAP)}개사 + KOSPI ({CALENDAR[0].strftime('%Y-%m-%d')} ~ {CALENDAR[-1].strftime('%Y-%m-%d')})")


In [ ]:
# ==============================================================================
# 4. Point-in-Time 1~8번 품질 판정 + 7개 핵심 알파 팩터 패널 구축 (미래정보 0%)
# ==============================================================================
def get_val(frame: dict, col: str, y: int):
    v = (frame.get(str(y)) or {}).get(col)
    return None if v is None else float(v)

def cagr_3y(vals_early, vals_late, span: float):
    if len(vals_early) < 2 or len(vals_late) < 2:
        return None
    a, b = sum(vals_early) / len(vals_early), sum(vals_late) / len(vals_late)
    return (b / a) ** (1.0 / span) - 1.0 if (a > 0 and b > 0) else None

def extract_company_year_factors(comp: dict, fy: int, rebal_dt: pd.Timestamp) -> dict | None:
    """매년 4월 첫 영업일(rebal_dt) 시점에 오직 과거 공시(<=fy)·과거 시세(<rebal_dt)로만 품질 및 7개 팩터 계산."""
    t = comp["ticker"]
    px_df = PRICE_MAP.get(t)
    if px_df is None or px_df.empty:
        return None
    hist = px_df.loc[:rebal_dt]
    if len(hist) < 120:
        return None
    px_now = float(hist["close"].iloc[-1])
    if px_now <= 0:
        return None

    frame = (comp.get("pit") or {}).get(str(fy))
    if not frame:
        return None
    path = comp.get("path") or "general"
    act_dps = {int(k): float(v) for k, v in (comp.get("act_dps") or {}).items() if v is not None}

    # --- 1~8번 현행 전략 A 품질 판정 (FY2018은 3년 창 폴백, FY2019+ 5년 창) ---
    dps_5 = [get_val(frame, "dps_common", y) for y in range(fy - 4, fy + 1)]
    paid_5 = sum(1 for v in dps_5 if v is not None and v > 0)
    cuts_5 = sum(1 for a, b in zip(dps_5, dps_5[1:]) if a is not None and b is not None and a > 0 and b < a * 0.99)
    if fy == 2018:
        g_dps = cagr_3y([v for y in range(2013, 2016) if (v := get_val(frame, "dps_common", y)) is not None],
                        [v for y in range(2016, 2019) if (v := get_val(frame, "dps_common", y)) is not None], 3.0)
        g_eps = cagr_3y([v for y in range(2013, 2016) if (v := get_val(frame, "eps_reported", y)) is not None],
                        [v for y in range(2016, 2019) if (v := get_val(frame, "eps_reported", y)) is not None], 3.0)
    else:
        g_dps = cagr_3y([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "dps_common", y)) is not None],
                        [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "dps_common", y)) is not None], 5.0)
        g_eps = cagr_3y([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "eps_reported", y)) is not None],
                        [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "eps_reported", y)) is not None], 5.0)

    v1 = "FAIL" if (paid_5 < 4 or cuts_5 >= 2) else ("PASS" if (paid_5 == 5 and cuts_5 == 0 and g_dps is not None and g_dps > 0) else "WARN")

    d3 = [get_val(frame, "dividends_paid", y) for y in range(fy - 2, fy + 1)]
    ni3 = [get_val(frame, "net_income_parent", y) for y in range(fy - 2, fy + 1)]
    tp3 = [get_val(frame, "treasury_purchase", y) or 0.0 for y in range(fy - 2, fy + 1)]
    p3 = (sum(d3) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None
    tr_3y = ((sum(d3) + sum(tp3)) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None

    if g_eps is None or g_dps is None: v2 = "NA"
    elif g_eps < 0 and g_dps > 0: v2 = "FAIL"
    elif g_eps > 0 and (g_dps <= g_eps + 0.03 or (p3 is not None and p3 < 30.0)): v2 = "PASS"
    else: v2 = "WARN"

    if path == "general":
        k3 = [v for y in range(fy - 4, fy + 1) if (v := get_val(frame, "fcf", y)) is not None]
        v3 = "FAIL" if sum(1 for v in k3 if v < 0) >= 2 else ("PASS" if (len(k3) >= 4 and sum(1 for v in k3 if v > 0) / len(k3) >= 0.8) else ("WARN" if len(k3) >= 3 else "NA"))
    else:
        eqs = [get_val(frame, "equity_parent", y) for y in range(fy - 4, fy + 1)]
        eq_ch = [b - a for a, b in zip(eqs, eqs[1:]) if a is not None and b is not None]
        v3 = "FAIL" if sum(1 for v in eq_ch if v < 0) >= 2 else ("PASS" if (len(eq_ch) >= 3 and sum(1 for v in eq_ch if v > 0) / len(eq_ch) >= 0.75) else ("WARN" if eq_ch else "NA"))

    sps = []
    for y in range(fy - 4, fy + 1):
        rf = RF_MAP.get(y, 0.025)
        coe = rf + 0.05
        if path != "general":
            roe = get_val(frame, "roe", y)
            if roe is not None: sps.append(roe - coe)
        else:
            roic = get_val(frame, "roic", y)
            if roic is not None:
                nd = get_val(frame, "net_debt", y)
                wacc = coe if (nd is None or nd <= 0) else (0.75 * coe + 0.25 * (rf + 0.015) * 0.78)
                sps.append(roic - wacc)
    avg5_sp = (sum(sps) / len(sps)) if sps else None
    p_thr = 0.03 if path == "general" else 0.01
    if len(sps) >= 3 and all(s < 0 for s in sps[-3:]): v4 = "FAIL"
    elif avg5_sp is not None: v4 = "PASS" if avg5_sp >= p_thr else "WARN"
    else: v4 = "NA"

    if tr_3y is None or p3 is None: v5 = "NA"
    elif tr_3y < 20.0 or p3 > 75.0 or (path != "general" and tr_3y > 100.0): v5 = "FAIL"
    elif tr_3y >= 30.0 and p3 <= 60.0 and (path == "general" or tr_3y <= 66.7): v5 = "PASS"
    else: v5 = "WARN"

    sh_old_y = 2015 if fy == 2018 else fy - 5
    s_now = get_val(frame, "outstanding_common", fy)
    s_old = get_val(frame, "outstanding_common", sh_old_y)
    sh_chg = ((s_now - s_old) / s_old) if (s_now and s_old and s_old > 0) else None
    canc5 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(fy - 4, fy + 1))
    acq5 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(fy - 4, fy + 1))
    if sh_chg is None: v6 = "NA"
    elif sh_chg >= 0.03: v6 = "FAIL"
    elif (sh_chg < -0.01 and canc5 > 0) or (-0.01 <= sh_chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)): v6 = "PASS"
    else: v6 = "WARN"

    if path != "general":
        v7, v8 = "참고", "PASS"
    else:
        fcfs3 = [get_val(frame, "fcf", y) for y in range(fy - 2, fy + 1)]
        if any(v is None for v in d3 + fcfs3): v7 = "NA"
        elif sum(fcfs3) <= 0: v7 = "FAIL" if (sum(d3) + sum(tp3)) > 0 else "PASS"
        else:
            cov = (sum(d3) + sum(tp3)) / sum(fcfs3)
            v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2/3 else "PASS")
        nd, nde, icr = get_val(frame, "net_debt", fy), get_val(frame, "net_debt_to_ebitda", fy), get_val(frame, "interest_coverage", fy)
        if nd is not None and nd <= 0: v8 = "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0): v8 = "FAIL"
        elif nde is not None and icr is not None: v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else: v8 = "NA"

    counted = [v1, v2, v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    n_fail, n_warn, n_na = counted.count("FAIL"), counted.count("WARN"), sum(1 for v in counted if v == "NA")
    pass_a = (n_fail == 0 and n_warn <= 2 and n_na == 0)
    if not pass_a:
        return None

    # --- 7개 핵심 알파 팩터 계산 (전부 매수 시점 rebal_dt 이전 정보만!) ---
    # 1) 밸류 할인율 (과거 연말 PER/PBR/배당률 중앙값 대비, 분할 보정 act_dps 사용)
    dps_y_adj = act_dps.get(fy)
    dps_y_rep = get_val(frame, "dps_common", fy)
    adj_ratio = (dps_y_adj / dps_y_rep) if (dps_y_adj and dps_y_rep and dps_y_rep > 0) else 1.0
    eps_y = get_val(frame, "eps_reported", fy)
    eq_y = get_val(frame, "equity_parent", fy)
    sh_tot = (get_val(frame, "outstanding_common", fy) or 0) + (get_val(frame, "outstanding_pref", fy) or 0)
    bps_y = (eq_y / sh_tot) if (eq_y and sh_tot > 0) else None

    c_per = px_now / (eps_y * adj_ratio) if (eps_y and eps_y > 0) else None
    c_pbr = px_now / (bps_y * adj_ratio) if (bps_y and bps_y > 0) else None
    c_yld = dps_y_adj / px_now if (dps_y_adj and dps_y_adj > 0) else None

    h_per, h_pbr, h_yld = [], [], []
    for y in range(max(2015, fy - 7), fy + 1):
        sub_y = px_df.loc[f"{y}-01-01":f"{y}-12-31"]
        if sub_y.empty: continue
        py = float(sub_y["close"].iloc[-1])
        ey = get_val(frame, "eps_reported", y)
        eq = get_val(frame, "equity_parent", y)
        st = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
        by = (eq / st) if (eq and st > 0) else None
        dy_a = act_dps.get(y)
        dy_r = get_val(frame, "dps_common", y)
        ar = (dy_a / dy_r) if (dy_a and dy_r and dy_r > 0) else adj_ratio
        if ey and ey > 0: h_per.append(py / (ey * ar))
        if by and by > 0: h_pbr.append(py / (by * ar))
        if dy_a and dy_a > 0: h_yld.append(dy_a / py)

    devs_raw, devs_cap = [], []
    cap_lim = LAB_CONFIG["VAL_CAP"]
    for cv, hv, hc in ((c_per, h_per, False), (c_pbr, h_pbr, False), (c_yld, h_yld, True)):
        if cv is not None and len(hv) >= 3:
            med = statistics.median(hv)
            if med > 0:
                d = (cv / med - 1.0) if hc else -(cv / med - 1.0)
                devs_raw.append(d)
                devs_cap.append(max(-cap_lim, min(cap_lim, d)))
    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None
    if val_cap is None:
        return None

    # 2~7) 모멘텀(12-1M), 200일선 추세, 저변동성, 유동성(60일 거래대금), 총환원+소각 점수
    closes = hist["close"]
    mom_12_1 = float(closes.iloc[-21] / closes.iloc[-252] - 1.0) if len(closes) >= 252 else (
        float(closes.iloc[-21] / closes.iloc[0] - 1.0) if len(closes) >= 60 else 0.0
    )
    ma200 = float(closes.tail(200).mean())
    trend_200 = (px_now / ma200 - 1.0) if ma200 > 0 else 0.0
    dret = closes.tail(252).pct_change().dropna()
    ann_vol = float(dret.std() * np.sqrt(252)) if len(dret) >= 30 else 0.30
    low_vol = -ann_vol   # 높을수록 저변동성
    turnover_60d_eok = float((hist["close"] * hist["volume"]).tail(60).mean()) / 1e8
    sh_yield = (tr_3y / 100.0 if tr_3y is not None else 0.0) - (sh_chg or 0.0)

    return {
        "fy": fy,
        "rebal_dt": rebal_dt.strftime("%Y-%m-%d"),
        "ticker": t,
        "name": comp["name"],
        "path": path,
        "is_fin": path != "general",
        "px_now": px_now,
        "ann_vol": ann_vol,
        "turnover_60d_eok": turnover_60d_eok,
        "dps_next_adj": act_dps.get(fy + 1, act_dps.get(fy, 0.0)),
        # 7개 표준화 대상 팩터 + 원형 밸류
        "f_val_raw": val_raw,
        "f_val": val_cap,
        "f_dps_g": g_dps if g_dps is not None else 0.0,
        "f_quality": avg5_sp if avg5_sp is not None else 0.0,
        "f_sh_yield": sh_yield,
        "f_mom": mom_12_1,
        "f_trend": trend_200,
        "f_lowvol": low_vol,
    }

# 연도별 4월 첫 영업일 및 다음 12개월 실제 세후 총수익률(FWD_1Y_TR) 계산
REBAL_DATES = {}
for fy in range(LAB_CONFIG["START_FY"], LAB_CONFIG["END_FY"] + 1):
    sub = CALENDAR[CALENDAR >= pd.Timestamp(f"{fy + 1}-04-01")]
    if len(sub) > 0:
        REBAL_DATES[fy] = sub[0]

PANEL_ROWS = []
sorted_fys = sorted(REBAL_DATES.keys())
for idx_f, fy in enumerate(sorted_fys):
    dt = REBAL_DATES[fy]
    next_dt = REBAL_DATES[sorted_fys[idx_f + 1]] if idx_f + 1 < len(sorted_fys) else CALENDAR[-1]
    k_ret = float(KOSPI_DF.loc[next_dt, "close"] / KOSPI_DF.loc[dt, "close"] - 1.0)
    for comp in COMPANIES:
        rec = extract_company_year_factors(comp, fy, dt)
        if rec is None:
            continue
        px_df = PRICE_MAP[rec["ticker"]]
        sub_fwd = px_df.loc[dt:next_dt]
        if sub_fwd.empty:
            continue
        p0, p1 = float(sub_fwd["close"].iloc[0]), float(sub_fwd["close"].iloc[-1])
        net_div = rec["dps_next_adj"] * (1.0 - LAB_CONFIG["DIV_TAX_RATE"])
        fwd_tr = (p1 + net_div) / p0 - 1.0
        rec["fwd_1y_tr"] = fwd_tr
        rec["fwd_1y_excess"] = fwd_tr - k_ret
        PANEL_ROWS.append(rec)

PANEL_DF = pd.DataFrame(PANEL_ROWS)
print(f"[패널 구축 완료] 총 {len(PANEL_DF)}개 기업-연도 (1~8번 원칙부합 표본) | 연도별 후보 수:")
print(PANEL_DF.groupby("rebal_dt")["ticker"].count().to_string())


In [ ]:
# ==============================================================================
# 5. [검증 1단계] 7개 핵심 팩터 예측력(Rank IC · ICIR · 상하위 격차) 분석
# ==============================================================================
FACTORS = [
    ("f_val",      "① 밸류 할인율 (±50% 캡)"),
    ("f_dps_g",    "② 정기 DPS 5년 성장률"),
    ("f_quality",  "③ 수익성 스프레드 (퀄리티)"),
    ("f_sh_yield", "④ 총주주환원 (배당+소각)"),
    ("f_mom",      "⑤ 중기 모멘텀 (12-1개월)"),
    ("f_trend",    "⑥ 200일선 추세 안정성"),
    ("f_lowvol",   "⑦ 저변동성 (리스크 방어)"),
]

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    df_xy = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df_xy) < 6:
        return np.nan
    rx = df_xy["x"].rank()
    ry = df_xy["y"].rank()
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(rx.corr(ry))

ic_rows = []
IC_BY_YEAR = {}
for fcol, flabel in FACTORS:
    y_ics, spreads = [], []
    per_yr = {}
    for dt_str, grp in PANEL_DF.groupby("rebal_dt"):
        # 유동성 10억 이상 표본 기준 검증
        g = grp[grp["turnover_60d_eok"] >= LAB_CONFIG["MIN_LIQ_EOK"]]
        if len(g) < 8:
            g = grp
        ic = spearman_ic(g[fcol], g["fwd_1y_tr"])
        per_yr[dt_str[:4]] = ic
        if not np.isnan(ic):
            y_ics.append(ic)
        n_q = max(3, int(len(g) * 0.30))
        top_r = float(g.nlargest(n_q, fcol)["fwd_1y_tr"].mean())
        bot_r = float(g.nsmallest(n_q, fcol)["fwd_1y_tr"].mean())
        spreads.append(top_r - bot_r)

    IC_BY_YEAR[fcol] = per_yr
    mean_ic = float(np.mean(y_ics)) if y_ics else 0.0
    std_ic = float(np.std(y_ics, ddof=1)) if len(y_ics) > 1 else 1.0
    icir = mean_ic / std_ic if std_ic > 0 else 0.0
    pos_rate = float(np.mean([v > 0 for v in y_ics])) if y_ics else 0.0
    avg_spread = float(np.mean(spreads)) if spreads else 0.0

    if mean_ic >= 0.06 and pos_rate >= 0.55:
        verdict = "유효 알파"
    elif mean_ic > 0.0:
        verdict = "약한 양(+)"
    else:
        verdict = "예측력 없음"

    ic_rows.append({
        "팩터": flabel,
        "평균 Rank IC": f"{mean_ic:+.3f}",
        "ICIR (안정성)": f"{icir:+.2f}",
        "양(+)의 연도 비율": f"{pos_rate:.0%} ({sum(v > 0 for v in y_ics)}/{len(y_ics)})",
        "상위30%-하위30% 격차": f"{avg_spread:+.1%}",
        "판정": verdict,
        "_col": fcol, "_mean_ic": mean_ic, "_icir": icir, "_spread": avg_spread, "_pos": pos_rate,
    })

IC_DF = pd.DataFrame(ic_rows).sort_values("_mean_ic", ascending=False).reset_index(drop=True)
print("■ [검증 1단계] 7개 핵심 팩터의 다음 1년 수익률 예측력 (Rank IC · 상하위 30% 격차)")
display(IC_DF[[c for c in IC_DF.columns if not c.startswith("_")]])


In [ ]:
# ==============================================================================
# 6. [검증 2단계] Walk-Forward(전진 검증) 학습 & 5종 포트폴리오 일별 백테스트
# ==============================================================================
def zscore_rank(s: pd.Series) -> pd.Series:
    """순위 백분위(0~1) 변환 — 극단값(이상치)에 휘둘리지 않는 안정적 표준화."""
    return s.rank(pct=True, method="average").fillna(0.5)

def run_daily_portfolio(weights_by_date: dict[pd.Timestamp, dict[str, float]], label: str) -> dict:
    """정확한 일별 수정주가 + 세후 배당 재투자(DRIP) + 편도 거래비용 0.25% 시뮬레이터."""
    rebal_set = sorted(weights_by_date.keys())
    start_dt = rebal_set[0]
    cal = CALENDAR[CALENDAR >= start_dt]
    fee = LAB_CONFIG["FEE_RATE"]
    div_tax = LAB_CONFIG["DIV_TAX_RATE"]

    # 배당금 맵 (연도별 4월 첫 영업일 지급 근사)
    act_dps_map = {c["ticker"]: {int(k): float(v) for k, v in (c.get("act_dps") or {}).items() if v is not None} for c in COMPANIES}
    rebal_lookup = set(rebal_set)

    shares: dict[str, float] = {}
    cash = float(LAB_CONFIG["INITIAL_CAPITAL"])
    nav_list, to_log = [], []
    div_paid_years = set()

    for dt in cal:
        # 세후 배당금 수취 (매년 4월 리밸런싱 시점)
        if dt in rebal_lookup and shares and dt.year not in div_paid_years:
            gross = sum(sh * act_dps_map.get(tk, {}).get(dt.year - 1, 0.0) for tk, sh in shares.items())
            cash += gross * (1.0 - div_tax)
            div_paid_years.add(dt.year)

        # 현재 시가 평가
        stock_val = 0.0
        px_today = {}
        for tk, sh in shares.items():
            sub = PRICE_MAP[tk].loc[:dt]
            px = float(sub["close"].iloc[-1]) if not sub.empty else 0.0
            px_today[tk] = px
            stock_val += sh * px
        total_val = stock_val + cash

        # 리밸런싱 집행
        if dt in rebal_lookup:
            tgt_w = weights_by_date[dt]
            valid_w = {}
            for tk, w in tgt_w.items():
                if tk in PRICE_MAP:
                    sub = PRICE_MAP[tk].loc[:dt]
                    if not sub.empty and float(sub["close"].iloc[-1]) > 0:
                        px_today[tk] = float(sub["close"].iloc[-1])
                        valid_w[tk] = w
            w_sum = sum(valid_w.values())
            if w_sum > 0:
                valid_w = {tk: w / w_sum for tk, w in valid_w.items()}
                old_val = {tk: shares.get(tk, 0.0) * px_today.get(tk, 0.0) for tk in set(shares) | set(valid_w)}
                traded = sum(abs(total_val * valid_w.get(tk, 0.0) - old_val.get(tk, 0.0)) for tk in old_val)
                cost = traded * fee
                to_log.append((dt.strftime("%Y-%m-%d"), traded / (2.0 * total_val) if total_val > 0 else 0.0))
                investable = max(0.0, total_val - cost)
                shares = {tk: (investable * w) / px_today[tk] for tk, w in valid_w.items()}
                cash = 0.0
                total_val = investable

        nav_list.append((dt, total_val))

    nav = pd.Series([v for _, v in nav_list], index=[d for d, _ in nav_list], name=label)
    dd = nav / nav.cummax() - 1.0
    return {"label": label, "nav": nav, "dd": dd, "turnover": to_log}

# --- 5가지 전략 비중 생성 (전부 100% Point-in-Time!) ---
TOP_N = LAB_CONFIG["TOP_N"]
MIN_LIQ = LAB_CONFIG["MIN_LIQ_EOK"]
FACTOR_COLS = [c for c, _ in FACTORS]

W_STRAT_A = {}          # ① 현행 전략 A (밸류원형 + DPS성장 50:50, 동일가중)
W_A_MOM = {}            # ② A + 유동성(10억) + 모멘텀(가치함정 차단)
W_MULTI_FIXED = {}      # ③ 다팩터 고정 (밸류 30% + 배당성장 25% + 퀄리티 20% + 모멘텀 25%, 학습 없음!)
W_WALK_FWD = {}         # ④ Walk-Forward 전진 학습형 (매년 과거 IC만 학습 -> 수축 -> 미래 해 검증)
W_MULTI_LOWVOL = {}     # ⑤ 다팩터 + 저변동성 비중조절 (1/vol 리스크 패리티)
WF_WEIGHT_HISTORY = []  # 연도별 Walk-Forward가 실제로 학습한 가중치 투명 공개!

rebal_strs = sorted(PANEL_DF["rebal_dt"].unique())
for idx_r, dt_str in enumerate(rebal_strs):
    dt = pd.Timestamp(dt_str)
    g_all = PANEL_DF[PANEL_DF["rebal_dt"] == dt_str].copy()
    g_liq = g_all[g_all["turnover_60d_eok"] >= MIN_LIQ].copy()
    if len(g_liq) < TOP_N:
        g_liq = g_all.copy()

    # ① 현행 전략 A (원형)
    g_all["sc_a"] = zscore_rank(g_all["f_val_raw"]) + zscore_rank(g_all["f_dps_g"])
    p_a = g_all.nlargest(TOP_N, "sc_a")
    W_STRAT_A[dt] = {t: 1.0 / len(p_a) for t in p_a["ticker"]}

    # ② A + 유동성 + 모멘텀 (밸류 35% + 배당성장 35% + 모멘텀 30%)
    g_liq["sc_amom"] = 0.35 * zscore_rank(g_liq["f_val"]) + 0.35 * zscore_rank(g_liq["f_dps_g"]) + 0.30 * zscore_rank(g_liq["f_mom"])
    p_am = g_liq.nlargest(TOP_N, "sc_amom")
    W_A_MOM[dt] = {t: 1.0 / len(p_am) for t in p_am["ticker"]}

    # ③ 다팩터 고정 (경제 논리 고정 가중치: 학습 0% -> 과적합 0%!)
    g_liq["sc_mf"] = (
        0.30 * zscore_rank(g_liq["f_val"]) +
        0.25 * zscore_rank(g_liq["f_dps_g"]) +
        0.20 * zscore_rank(g_liq["f_quality"]) +
        0.25 * zscore_rank(g_liq["f_mom"])
    )
    p_mf = g_liq.nlargest(TOP_N, "sc_mf")
    W_MULTI_FIXED[dt] = {t: 1.0 / len(p_mf) for t in p_mf["ticker"]}

    # ⑤ 다팩터 + 저변동성 비중조절 (동일 종목 풀에 1/vol 비중 적용)
    inv_v = 1.0 / p_mf["ann_vol"].clip(0.12, 0.80)
    W_MULTI_LOWVOL[dt] = {t: float(w / inv_v.sum()) for t, w in zip(p_mf["ticker"], inv_v)}

    # ④ Walk-Forward 전진 학습형 (오직 dt_str 이전의 완료된 연도 데이터만 학습!)
    train_dts = rebal_strs[:idx_r]   # 현재 연도(dt_str)와 미래 연도는 절대 포함 금지!
    eq_w = {fc: 1.0 / len(FACTOR_COLS) for fc in FACTOR_COLS}
    if len(train_dts) < 2:
        # 초기 2년 워밍업: 균등 가중치 (미래 누출 원천 차단!)
        wf_w = dict(eq_w)
        train_note = "워밍업(균등)"
    else:
        raw_ic = {}
        for fc in FACTOR_COLS:
            ics = []
            for tdt in train_dts:
                tg = PANEL_DF[(PANEL_DF["rebal_dt"] == tdt) & (PANEL_DF["turnover_60d_eok"] >= MIN_LIQ)]
                ic = spearman_ic(tg[fc], tg["fwd_1y_tr"])
                if not np.isnan(ic):
                    ics.append(ic)
            # 양(+)의 과거 IC만 반영 (음수 팩터 0으로 클립)
            raw_ic[fc] = max(0.0, float(np.mean(ics))) if ics else 0.0
        s_ic = sum(raw_ic.values())
        learned_w = {fc: (raw_ic[fc] / s_ic if s_ic > 0 else eq_w[fc]) for fc in FACTOR_COLS}
        # Shrinkage 수축 + 단일 팩터 상한(40%) 적용 -> 과적합 차단!
        lam = LAB_CONFIG["WF_SHRINKAGE"]
        blended = {fc: min(LAB_CONFIG["WF_MAX_WEIGHT"], (1 - lam) * learned_w[fc] + lam * eq_w[fc]) for fc in FACTOR_COLS}
        sb = sum(blended.values())
        wf_w = {fc: blended[fc] / sb for fc in FACTOR_COLS}
        train_note = f"{train_dts[0][:4]}~{train_dts[-1][:4]} 학습"

    g_liq["sc_wf"] = sum(wf_w[fc] * zscore_rank(g_liq[fc]) for fc in FACTOR_COLS)
    p_wf = g_liq.nlargest(TOP_N, "sc_wf")
    W_WALK_FWD[dt] = {t: 1.0 / len(p_wf) for t in p_wf["ticker"]}
    WF_WEIGHT_HISTORY.append({"연도": dt_str[:4], "학습구간": train_note, **{fl.split()[1]: f"{wf_w[fc]:.0%}" for fc, fl in FACTORS}})

# 일별 백테스트 실행 + KOSPI PR & KOSPI 배당 포함(TR 추정) 벤치마크 생성
RESULTS = {
    "현행 전략 A (2팩터 기준선)":       run_daily_portfolio(W_STRAT_A, "현행 전략 A"),
    "A + 유동성·모멘텀 (가치함정 차단)": run_daily_portfolio(W_A_MOM, "A+모멘텀"),
    "다팩터 고정 (4팩터 경제논리)":     run_daily_portfolio(W_MULTI_FIXED, "다팩터 고정"),
    "Walk-Forward 자동 학습 (전진검증)": run_daily_portfolio(W_WALK_FWD, "Walk-Forward"),
    "다팩터 + 저변동성 비중 (리스크조절)": run_daily_portfolio(W_MULTI_LOWVOL, "다팩터+저변동"),
}

start_bt = RESULTS["현행 전략 A (2팩터 기준선)"]["nav"].index[0]
k_cal = KOSPI_DF.loc[start_bt:, "close"]
KOSPI_PR_NAV = k_cal / k_cal.iloc[0] * LAB_CONFIG["INITIAL_CAPITAL"]
# KOSPI 배당 포함(TR 추정): 일별 복리로 연 +1.8% 세후 배당 가산 (솔직한 비교 기준!)
days_elapsed = (k_cal.index - k_cal.index[0]).days / 365.25
KOSPI_TR_NAV = KOSPI_PR_NAV * ((1.0 + LAB_CONFIG["KOSPI_EST_DIV_YIELD"]) ** days_elapsed)

bt_yrs_list = sorted(set(k_cal.index.year))
RF_AVG = float(np.mean([RF_MAP.get(y, 0.028) for y in bt_yrs_list]))

def calc_perf(nav: pd.Series, to_log: list | None = None) -> dict:
    yrs = (nav.index[-1] - nav.index[0]).days / 365.25
    init = float(LAB_CONFIG["INITIAL_CAPITAL"])
    tot = float(nav.iloc[-1] / init - 1.0)
    cagr = float((nav.iloc[-1] / init) ** (1.0 / yrs) - 1.0)
    dd = nav / nav.cummax() - 1.0
    mdd = float(dd.min())
    vol = float(nav.pct_change().dropna().std() * np.sqrt(252))
    sharpe = (cagr - RF_AVG) / vol if vol > 0 else 0.0
    calmar = cagr / abs(mdd) if mdd < 0 else 0.0
    avg_to = float(np.mean([x[1] for x in to_log[1:]])) if (to_log and len(to_log) > 1) else np.nan
    return {"end_val": float(nav.iloc[-1]), "tot": tot, "cagr": cagr, "mdd": mdd, "vol": vol, "sharpe": sharpe, "calmar": calmar, "to": avg_to}

PERF_RAW = {k: calc_perf(v["nav"], v["turnover"]) for k, v in RESULTS.items()}
PERF_RAW["KOSPI (가격지수 PR)"] = calc_perf(KOSPI_PR_NAV)
PERF_RAW["KOSPI (배당포함 TR 추정)"] = calc_perf(KOSPI_TR_NAV)

PERF_TABLE = pd.DataFrame([
    {
        "전략": name,
        "최종 평가액 (1억)": f"{m['end_val']/1e8:.2f}억 원",
        "누적 수익률": f"{m['tot']:+.1%}",
        "CAGR (연평균)": f"{m['cagr']:+.2%}",
        "vs 코스피TR": f"{(m['cagr'] - PERF_RAW['KOSPI (배당포함 TR 추정)']['cagr'])*100:+.2f}%p",
        "MDD (최대낙폭)": f"{m['mdd']:.1%}",
        "변동성": f"{m['vol']:.1%}",
        "샤프지수": f"{m['sharpe']:.2f}",
        "칼마지수": f"{m['calmar']:.2f}",
        "연교체율": "-" if np.isnan(m["to"]) else f"{m['to']:.1%}",
    }
    for name, m in PERF_RAW.items()
])

print("■ [검증 2단계] 5종 전략 vs KOSPI PR · KOSPI TR(배당포함 추정) 일별 백테스트 성과표")
display(PERF_TABLE)
print("\n■ Walk-Forward 연도별 학습 가중치 이력 (과거 구간만 학습 -> 미래 해 적용)")
WF_HIST_DF = pd.DataFrame(WF_WEIGHT_HISTORY)
display(WF_HIST_DF)


In [ ]:
# ==============================================================================
# 7. [검증 3~4단계] 과적합 스트레스 테스트 (이어받기 몬테카를로 500회 + 대박주 제거 + 채택 관문)
# ==============================================================================
# 최고 성과 후보 전략 선정 (현행 A 제외한 4개 신규 후보 중 샤프지수 1위)
CAND_NAMES = [k for k in RESULTS.keys() if not k.startswith("현행")]
BEST_CAND_NAME = max(CAND_NAMES, key=lambda k: PERF_RAW[k]["sharpe"])
BEST_CAND_M = PERF_RAW[BEST_CAND_NAME]
BASE_A_M = PERF_RAW["현행 전략 A (2팩터 기준선)"]
K_TR_M = PERF_RAW["KOSPI (배당포함 TR 추정)"]

BEST_W_MAP = {
    "A + 유동성·모멘텀 (가치함정 차단)": W_A_MOM,
    "다팩터 고정 (4팩터 경제논리)":     W_MULTI_FIXED,
    "Walk-Forward 자동 학습 (전진검증)": W_WALK_FWD,
    "다팩터 + 저변동성 비중 (리스크조절)": W_MULTI_LOWVOL,
}[BEST_CAND_NAME]

# (A) 최고 대박 1종목 제거 테스트 (Leave-One-Best-Stock-Out)
stock_contrib = {}
for dt, wmap in BEST_W_MAP.items():
    dt_str = dt.strftime("%Y-%m-%d")
    sub_p = PANEL_DF[(PANEL_DF["rebal_dt"] == dt_str) & (PANEL_DF["ticker"].isin(wmap.keys()))]
    for _, r in sub_p.iterrows():
        stock_contrib[r["ticker"]] = stock_contrib.get(r["ticker"], 0.0) + float(r["fwd_1y_tr"]) * wmap[r["ticker"]]
top_hero_tk = max(stock_contrib, key=stock_contrib.get)
top_hero_nm = PANEL_DF.loc[PANEL_DF["ticker"] == top_hero_tk, "name"].iloc[0]

w_loo = {}
for dt, wmap in BEST_W_MAP.items():
    rem = {t: w for t, w in wmap.items() if t != top_hero_tk}
    s = sum(rem.values())
    w_loo[dt] = {t: w / s for t, w in rem.items()} if s > 0 else wmap
RES_LOO = run_daily_portfolio(w_loo, "LOO")
LOO_M = calc_perf(RES_LOO["nav"], RES_LOO["turnover"])

# (B) 무작위 10종목 500회 몬테카를로 시뮬레이션 (50회마다 Drive 체크포인트 저장 -> 끊겨도 이어받기!)
CKPT_MC_NAME = f"ckpt_montecarlo_{LAB_CONFIG['START_FY']}_{LAB_CONFIG['END_FY']}_n{LAB_CONFIG['TOP_N']}.json.gz"
mc_state = load_ckpt_json(CKPT_MC_NAME, force=LAB_CONFIG["FORCE_REFRESH_MC"])
mc_cagrs = list(mc_state.get("cagrs") or [])
mc_sharpes = list(mc_state.get("sharpes") or [])
target_mc = LAB_CONFIG["MC_RUNS"]

if len(mc_cagrs) >= target_mc:
    print(f"[몬테카를로 이어받기 완료] 기존 Drive 완료분 {len(mc_cagrs)}회 재사용!")
else:
    print(f"[몬테카를로 스트레스 테스트] 완료 {len(mc_cagrs)}/{target_mc}회 -> 남은 {target_mc - len(mc_cagrs)}회 실행 중 (50회마다 Drive 저장)...")
    pool_by_dt = {pd.Timestamp(dt_str): grp["ticker"].tolist() for dt_str, grp in PANEL_DF.groupby("rebal_dt")}
    t0 = time.time()
    try:
        for run_i in range(len(mc_cagrs), target_mc):
            rng = random.Random(LAB_CONFIG["MC_SEED"] + run_i)
            w_rnd = {}
            for dt, tks in pool_by_dt.items():
                chosen = rng.sample(tks, min(TOP_N, len(tks)))
                w_rnd[dt] = {tk: 1.0 / len(chosen) for tk in chosen}
            m_r = calc_perf(run_daily_portfolio(w_rnd, "rnd")["nav"])
            mc_cagrs.append(m_r["cagr"])
            mc_sharpes.append(m_r["sharpe"])
            if len(mc_cagrs) % 50 == 0 or len(mc_cagrs) == target_mc:
                save_ckpt_json(CKPT_MC_NAME, {"cagrs": mc_cagrs, "sharpes": mc_sharpes})
                print(f"  몬테카를로 진행 {len(mc_cagrs)}/{target_mc}회 ({time.time()-t0:.1f}초, Drive 저장됨)")
    finally:
        save_ckpt_json(CKPT_MC_NAME, {"cagrs": mc_cagrs, "sharpes": mc_sharpes})

mc_cagrs_arr = np.array(mc_cagrs[:target_mc])
mc_sharpes_arr = np.array(mc_sharpes[:target_mc])
p_val_cagr = float(np.mean(mc_cagrs_arr >= BEST_CAND_M["cagr"]))
p_val_sharpe = float(np.mean(mc_sharpes_arr >= BEST_CAND_M["sharpe"]))
mc_median_cagr = float(np.median(mc_cagrs_arr))

# (C) 보유 종목 수 민감도 (TOP_N = 8, 10, 12, 15) — 다팩터 고정 모델 기준
SENS_ROWS = []
for n_test in (8, 10, 12, 15):
    w_n = {}
    for dt_str in rebal_strs:
        dt = pd.Timestamp(dt_str)
        g_l = PANEL_DF[(PANEL_DF["rebal_dt"] == dt_str) & (PANEL_DF["turnover_60d_eok"] >= MIN_LIQ)].copy()
        if len(g_l) < n_test:
            g_l = PANEL_DF[PANEL_DF["rebal_dt"] == dt_str].copy()
        g_l["sc"] = 0.30 * zscore_rank(g_l["f_val"]) + 0.25 * zscore_rank(g_l["f_dps_g"]) + 0.20 * zscore_rank(g_l["f_quality"]) + 0.25 * zscore_rank(g_l["f_mom"])
        p_n = g_l.nlargest(n_test, "sc")
        w_n[dt] = {t: 1.0 / len(p_n) for t in p_n["ticker"]}
    m_n = calc_perf(run_daily_portfolio(w_n, f"N={n_test}")["nav"])
    SENS_ROWS.append({
        "종목 수 (TOP_N)": f"{n_test}종목",
        "CAGR": f"{m_n['cagr']:+.2%}",
        "vs 코스피TR": f"{(m_n['cagr'] - K_TR_M['cagr'])*100:+.2f}%p",
        "MDD": f"{m_n['mdd']:.1%}",
        "샤프지수": f"{m_n['sharpe']:.2f}",
    })
SENS_DF = pd.DataFrame(SENS_ROWS)

# (D) 엄격한 실전 채택 4중 관문 (Go / Hold / Reject Gate)
gate1 = (BEST_CAND_M["cagr"] > BASE_A_M["cagr"] + 0.005) and (BEST_CAND_M["cagr"] > K_TR_M["cagr"])
gate2 = (BEST_CAND_M["sharpe"] >= BASE_A_M["sharpe"] + 0.03) and (BEST_CAND_M["mdd"] >= BASE_A_M["mdd"] - 0.03)
gate3 = (LOO_M["cagr"] > BASE_A_M["cagr"] - 0.005)
gate4 = (p_val_sharpe <= 0.20)
gates_passed = sum([gate1, gate2, gate3, gate4])

GATE_DF = pd.DataFrame([
    {"검증 관문": "① 수익성 우위 (현행 A + 코스피TR 동시 초과)", "실측값": f"후보 {BEST_CAND_M['cagr']:+.2%} vs A {BASE_A_M['cagr']:+.2%} / TR {K_TR_M['cagr']:+.2%}", "판정": "통과" if gate1 else "미달"},
    {"검증 관문": "② 위험 효율 (샤프 +0.03↑ & MDD 방어)", "실측값": f"샤프 {BEST_CAND_M['sharpe']:.2f} (A {BASE_A_M['sharpe']:.2f}) · MDD {BEST_CAND_M['mdd']:.1%}", "판정": "통과" if gate2 else "미달"},
    {"검증 관문": f"③ 대박주 의존 배제 ({top_hero_nm} 1종목 제외)", "실측값": f"제외 후 CAGR {LOO_M['cagr']:+.2%} (기존 A {BASE_A_M['cagr']:+.2%})", "판정": "통과" if gate3 else "미달"},
    {"검증 관문": f"④ 통계 유의성 (무작위 {target_mc}회 상위 20% 이내)", "실측값": f"샤프 상위 {p_val_sharpe:.1%} (CAGR 상위 {p_val_cagr:.1%}, 무작위 중앙 {mc_median_cagr:+.2%})", "판정": "통과" if gate4 else "미달"},
])

if gates_passed == 4:
    FINAL_VERDICT = f"채택 권장 (4/4 관문 통과): '{BEST_CAND_NAME}'을 실전 스크리너 순위 공식에 반영할 통계적 근거가 확인되었습니다."
    VERDICT_BADGE = "채택 권장"
elif gates_passed >= 2:
    FINAL_VERDICT = f"조건부 참고 ({gates_passed}/4 관문 통과): '{BEST_CAND_NAME}'이 일부 지표를 개선했으나 완전 통과는 아니므로, 현행 전략 A를 기본으로 두고 보조 참고 지표로만 활용하세요."
    VERDICT_BADGE = "조건부 참고"
else:
    FINAL_VERDICT = f"보류 / 현행 유지 ({gates_passed}/4 관문 통과): 과적합 위험을 고려할 때 섣불리 규칙을 바꾸기보다 현행 전략 A 유지가 안전합니다."
    VERDICT_BADGE = "현행 A 유지"

print(f"\n■ [검증 4단계] 최우수 후보 '{BEST_CAND_NAME}' 과적합 차단 4중 관문 결과 ({gates_passed}/4 통과)")
display(GATE_DF)
print(f"  => 최종 판정: {FINAL_VERDICT}")
display(SENS_DF)


In [ ]:
# ==============================================================================
# 8. 시각화 대시보드 & A4 PDF 리포트 생성 · Google Drive 누적 저장 (브라우저 다운 생략)
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[tuple[str, str]] | None = None) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"  [참고] WeasyPrint 건너뜀({type(e).__name__}) -> Matplotlib PDF 폴백")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for title, b64 in (chart_b64_list or []):
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img); plt.axis("off"); plt.title(title, fontsize=13, fontweight="bold")
                plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception as e2:
        print(f"  [주의] PDF 생성 실패: {e2}")
        return False

as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

# 4패널 연구실 대시보드 차트
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15.4, 8.8), facecolor="white")
fig.suptitle("한국 배당성장주 알파 팩터 연구실 · 과적합 차단 4중 검증 대시보드", fontsize=15.5, fontweight="bold", color="#0F2922", y=0.985)
fig.text(0.5, 0.946, f"검증 기간 {start_bt.strftime('%Y-%m-%d')} ~ {CALENDAR[-1].strftime('%Y-%m-%d')}  |  374개사 PIT  |  KOSPI TR 추정(연 +{LAB_CONFIG['KOSPI_EST_DIV_YIELD']:.1%}) 포함  |  몬테카를로 {LAB_CONFIG['MC_RUNS']}회", ha="center", fontsize=9.3, color="#64748B")

# (1) 누적 수익률 곡선 (5종 전략 vs KOSPI TR 추정)
palette = ["#1D4ED8", "#0F766E", "#059669", "#7C3AED", "#D97706"]
for (nm, res), col in zip(RESULTS.items(), palette):
    ret = (res["nav"] / LAB_CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
    lw = 2.6 if nm == BEST_CAND_NAME else (2.1 if nm.startswith("현행") else 1.5)
    ax1.plot(ret.index, ret, label=f"{res['label']} ({ret.iloc[-1]:+.0f}%)", color=col, lw=lw, alpha=0.92)
k_tr_ret = (KOSPI_TR_NAV / LAB_CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ax1.plot(k_tr_ret.index, k_tr_ret, label=f"KOSPI TR추정 ({k_tr_ret.iloc[-1]:+.0f}%)", color="#64748B", lw=1.8, ls="--")
ax1.set_title("① 전략별 일별 누적 수익률 추이 (배당 재투자 vs 코스피 TR 추정)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.legend(loc="upper left", fontsize=8.0, frameon=True, facecolor="white", edgecolor="#E2E8F0", ncol=2)
ax1.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# (2) 7개 팩터 평균 Rank IC 막대 차트
ic_plot = IC_DF.iloc[::-1]
cols_ic = ["#059669" if v >= 0.06 else ("#38BDF8" if v > 0 else "#EF4444") for v in ic_plot["_mean_ic"]]
bars2 = ax2.barh(ic_plot["팩터"], ic_plot["_mean_ic"], color=cols_ic, height=0.58)
ax2.axvline(0, color="#334155", lw=0.9)
for b, v in zip(bars2, ic_plot["_mean_ic"]):
    ax2.text(v + (0.006 if v >= 0 else -0.006), b.get_y() + b.get_height() / 2, f"{v:+.3f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=8.5, fontweight="bold", color="#0F172A")
ax2.set_title("② 7개 팩터의 다음 1년 수익률 예측력 (평균 Rank IC)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax2.set_xlim(min(-0.15, ic_plot["_mean_ic"].min() - 0.06), max(0.25, ic_plot["_mean_ic"].max() + 0.08))
ax2.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.7); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

# (3) 몬테카를로 500회 무작위 포트폴리오 CAGR 분포 vs 우리 전략 위치
ax3.hist(mc_cagrs_arr * 100, bins=26, color="#CBD5E1", edgecolor="white", alpha=0.9, label=f"무작위 10종목 {len(mc_cagrs_arr)}회 분포")
ax3.axvline(BASE_A_M["cagr"] * 100, color="#1D4ED8", lw=2.2, ls="--", label=f"현행 전략 A ({BASE_A_M['cagr']:+.2%})")
ax3.axvline(BEST_CAND_M["cagr"] * 100, color="#059669", lw=2.5, label=f"최우수 후보 ({BEST_CAND_M['cagr']:+.2%}, 상위 {p_val_cagr:.1%})")
ax3.axvline(K_TR_M["cagr"] * 100, color="#DC2626", lw=1.8, ls=":", label=f"KOSPI TR추정 ({K_TR_M['cagr']:+.2%})")
ax3.set_title("③ 과적합 검증: 무작위 500회 추출 대비 전략의 통계적 위치 (CAGR)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax3.set_xlabel("연평균 수익률 (CAGR, %)"); ax3.set_ylabel("빈도")
ax3.legend(loc="upper left", fontsize=8.1, frameon=True, facecolor="white", edgecolor="#E2E8F0")
ax3.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

# (4) 위험(변동성) vs 수익(CAGR) 효율 맵
for name, m in PERF_RAW.items():
    is_k = name.startswith("KOSPI")
    is_best = (name == BEST_CAND_NAME)
    col = "#DC2626" if is_k else ("#059669" if is_best else ("#1D4ED8" if name.startswith("현행") else "#64748B"))
    mk = "D" if is_k else ("*" if is_best else "o")
    sz = 230 if is_best else (110 if name.startswith("현행") or is_k else 85)
    ax4.scatter(m["vol"] * 100, m["cagr"] * 100, s=sz, c=col, marker=mk, edgecolors="#0F172A", lw=0.8, zorder=5 if is_best else 3)
    short_nm = name.split("(")[0].strip()
    ax4.annotate(f"{short_nm}\n(샤프 {m['sharpe']:.2f})", (m["vol"] * 100, m["cagr"] * 100), xytext=(6, 4), textcoords="offset points", fontsize=8.0, fontweight="bold" if is_best else "normal", color="#0F172A")
ax4.set_title("④ 위험(변동성) 대비 수익률(CAGR) 효율 지도 (왼쪽 위 = 최고)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax4.set_xlabel("연환산 변동성 (%) — 낮을수록 안정"); ax4.set_ylabel("연평균 수익률 CAGR (%)")
ax4.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax4.spines["top"].set_visible(False); ax4.spines["right"].set_visible(False)

fig.subplots_adjust(top=0.90, bottom=0.07, left=0.07, right=0.98, hspace=0.33, wspace=0.22)
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=155, bbox_inches="tight", facecolor="white")
DASH_B64 = base64.b64encode(buf.getvalue()).decode()
plt.show()
plt.close(fig)

def lab_badge(val: str) -> str:
    s = str(val)
    if s in ("통과", "유효 알파", "채택 권장"):
        return f'<span class="pill pill-green">{escape(s)}</span>'
    if s in ("약한 양(+)", "조건부 참고"):
        return f'<span class="pill pill-amber">{escape(s)}</span>'
    if s in ("미달", "예측력 없음", "현행 A 유지"):
        return f'<span class="pill pill-red">{escape(s)}</span>'
    return escape(s)

def render_tbl(df: pd.DataFrame) -> str:
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    trs = []
    for _, row in df.iterrows():
        tds = []
        for c in df.columns:
            cls = "col-left" if c in ("전략", "팩터", "검증 관문", "실측값", "학습구간") else ""
            val = lab_badge(row[c]) if c == "판정" else escape(str(row[c]))
            tds.append(f'<td class="{cls}">{val}</td>')
        trs.append("<tr>" + "".join(tds) + "</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

def _nanum_css() -> str:
    parts = []
    for w, p in [(400, "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"), (700, "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf")]:
        if os.path.exists(p):
            parts.append(f"@font-face {{ font-family:'NanumPDF'; src:url('file://{p}'); font-weight:{w}; }}")
    return "\n  ".join(parts)

ic_view = IC_DF[[c for c in IC_DF.columns if not c.startswith("_")]]
html_report = f"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<title>한국 배당성장주 알파 팩터 연구실 리포트 ({as_of_str})</title>
<style>
  {_nanum_css()}
  @page {{
    size: A4 landscape;
    margin: 10mm 11mm 12mm 11mm;
    @bottom-left {{ content: "한국 배당성장주 알파 팩터 연구실 · 과적합 차단 전진검증 ({as_of_str})"; font-family:'NanumPDF',sans-serif; font-size:7.5pt; color:#94a3b8; }}
    @bottom-right {{ content: "Page " counter(page) " / " counter(pages); font-family:'NanumPDF',sans-serif; font-size:7.5pt; color:#94a3b8; }}
  }}
  :root {{ --forest:#0F2922; --teal:#134E4A; --accent:#0F766E; --gold:#D4A34A; --ink:#0F172A; --muted:#64748B; --bg:#F8FAFC; --card:#FFFFFF; --line:#E2E8F0; }}
  * {{ box-sizing:border-box; }}
  body {{ margin:0; padding:12px; background:var(--bg); color:var(--ink); font-family:'NanumPDF','NanumGothic',sans-serif; font-size:10.8px; line-height:1.5; font-variant-numeric:tabular-nums; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .masthead {{ background:linear-gradient(135deg, var(--forest) 0%, var(--teal) 100%); color:#fff; padding:17px 22px; border-radius:12px; margin-bottom:11px; border-bottom:4px solid var(--gold); }}
  .eyebrow {{ font-size:8.8px; letter-spacing:1.5px; color:#FDE68A; font-weight:700; text-transform:uppercase; margin-bottom:4px; }}
  .masthead h1 {{ margin:0 0 4px; font-size:19px; font-weight:700; }}
  .meta-row {{ display:flex; flex-wrap:wrap; gap:6px; margin-top:8px; }}
  .chip {{ background:rgba(255,255,255,0.13); border:1px solid rgba(255,255,255,0.22); padding:2.5px 9px; border-radius:999px; font-size:9.3px; color:#F1F5F9; }}
  .kpis {{ display:grid; grid-template-columns:repeat(4, 1fr); gap:10px; margin-bottom:11px; break-inside:avoid; }}
  .kpi {{ background:var(--card); border:1px solid var(--line); border-top:3.5px solid var(--accent); border-radius:10px; padding:10px 13px; }}
  .kpi .lab {{ font-size:9.5px; color:var(--muted); font-weight:700; }}
  .kpi .val {{ font-size:17px; font-weight:700; margin-top:2px; }}
  .kpi .sub {{ font-size:8.8px; color:var(--muted); margin-top:2px; }}
  .card {{ background:var(--card); border:1px solid var(--line); border-radius:10px; padding:12px 14px; margin-bottom:10px; break-inside:avoid; }}
  .sec-title {{ display:flex; align-items:center; gap:8px; margin:0 0 8px; font-size:12.5px; font-weight:700; color:var(--forest); }}
  .sec-num {{ display:inline-block; background:var(--forest); color:#fff; font-size:9px; padding:2px 7px; border-radius:5px; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.5px; }}
  table.tbl th {{ background:#F1F5F9; color:#334155; font-weight:700; padding:5.5px 5px; border-top:1.5px solid var(--forest); border-bottom:1px solid #CBD5E1; text-align:right; }}
  table.tbl td {{ padding:5px 5px; border-bottom:1px solid #EEF2F6; text-align:right; }}
  table.tbl tbody tr:nth-child(even) {{ background:#F9FEFC; }}
  table.tbl td.col-left, table.tbl th:first-child {{ text-align:left; font-weight:700; }}
  .pill {{ display:inline-block; padding:1.5px 6.5px; border-radius:999px; font-size:8.2px; font-weight:700; white-space:nowrap; }}
  .pill-green {{ background:#DCFCE7; color:#166534; }}
  .pill-amber {{ background:#FEF3C7; color:#92400E; }}
  .pill-red {{ background:#FEE2E2; color:#991B1B; }}
  img.chart {{ width:100%; height:auto; display:block; border-radius:6px; }}
  .page-break {{ page-break-before:always; break-before:page; }}
  .foot {{ font-size:8.6px; color:var(--muted); border-top:1px solid var(--line); padding-top:6px; }}
</style>
</head>
<body>
<div class="wrap">
  <div class="masthead">
    <div class="eyebrow">Alpha Factor Lab · Anti-Overfitting Walk-Forward Validation</div>
    <h1>한국 배당성장주 알파 팩터 연구실 · 과적합 차단 4중 검증 리포트</h1>
    <div class="meta-row">
      <span class="chip">검증 기간 {start_bt.strftime('%Y-%m-%d')} ~ {CALENDAR[-1].strftime('%Y-%m-%d')}</span>
      <span class="chip">공개 PIT 374개사</span>
      <span class="chip">전진검증(Walk-Forward) 수축 {LAB_CONFIG['WF_SHRINKAGE']:.0%}</span>
      <span class="chip">몬테카를로 {LAB_CONFIG['MC_RUNS']}회</span>
      <span class="chip">최종 판정: {VERDICT_BADGE}</span>
    </div>
  </div>

  <div class="kpis">
    <div class="kpi"><div class="lab">현행 전략 A (기준선)</div><div class="val">{BASE_A_M['cagr']:+.2%} /년</div><div class="sub">누적 {BASE_A_M['tot']:+.1%} · MDD {BASE_A_M['mdd']:.1%} · 샤프 {BASE_A_M['sharpe']:.2f}</div></div>
    <div class="kpi" style="border-top-color:#059669"><div class="lab">최우수 후보 ({escape(BEST_CAND_NAME.split('(')[0].strip())})</div><div class="val" style="color:#059669">{BEST_CAND_M['cagr']:+.2%} /년</div><div class="sub">누적 {BEST_CAND_M['tot']:+.1%} · MDD {BEST_CAND_M['mdd']:.1%} · 샤프 {BEST_CAND_M['sharpe']:.2f}</div></div>
    <div class="kpi" style="border-top-color:#64748B"><div class="lab">KOSPI 배당포함 TR 추정</div><div class="val">{K_TR_M['cagr']:+.2%} /년</div><div class="sub">PR {PERF_RAW['KOSPI (가격지수 PR)']['cagr']:+.2%} + 연 {LAB_CONFIG['KOSPI_EST_DIV_YIELD']:.1%} 배당 가산</div></div>
    <div class="kpi" style="border-top-color:#D97706"><div class="lab">과적합 차단 4중 관문</div><div class="val">{gates_passed} / 4 통과</div><div class="sub">무작위 {LAB_CONFIG['MC_RUNS']}회 중 샤프 상위 {p_val_sharpe:.1%}</div></div>
  </div>

  <div class="card" style="background:#F0FDF9;border-color:#A7F3D0;border-left:4px solid #0F766E;">
    <div class="sec-title" style="color:#065F46;"><span class="sec-num" style="background:#0F766E;">결론</span>과적합 차단 4중 검증 최종 판정</div>
    <p style="margin:0 0 6px;font-size:10.5px;color:#064E3B;"><b>{escape(FINAL_VERDICT)}</b></p>
    <p style="margin:0;font-size:9.3px;color:#047857;">
      ※ 최고 대박 1종목(<b>{escape(top_hero_nm)}</b>)을 빼고 다시 계산해도 후보 전략 CAGR은 <b>{LOO_M['cagr']:+.2%}</b>(현행 A {BASE_A_M['cagr']:+.2%})였으며,
      원칙 부합 종목에서 무작위로 10종목을 뽑은 {LAB_CONFIG['MC_RUNS']}회 중앙값 CAGR은 <b>{mc_median_cagr:+.2%}</b>였습니다.
    </p>
  </div>

  <div class="card">
    <div class="sec-title"><span class="sec-num">01</span>과적합 차단 4중 관문 스코어카드 (최우수 후보: {escape(BEST_CAND_NAME)})</div>
    {render_tbl(GATE_DF)}
  </div>

  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">02</span>4패널 알파 팩터 · Walk-Forward · 몬테카를로 검증 대시보드</div>
    <img class="chart" src="data:image/png;base64,{DASH_B64}" alt="알파 연구실 대시보드">
  </div>

  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">03</span>5종 전략 vs KOSPI PR · KOSPI TR(배당 포함 추정) 성과 비교표</div>
    {render_tbl(PERF_TABLE)}
  </div>

  <div class="card">
    <div class="sec-title"><span class="sec-num">04</span>7개 핵심 팩터 예측력 분해 (Rank IC · ICIR · 상하위 30% 수익률 격차)</div>
    {render_tbl(ic_view)}
  </div>

  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">05</span>Walk-Forward 연도별 학습 가중치 투명 공개 (미래정보 0%) &amp; 종목 수(TOP_N) 민감도</div>
    <div style="display:grid;grid-template-columns:1.45fr 1fr;gap:12px;">
      <div>{render_tbl(WF_HIST_DF)}</div>
      <div>{render_tbl(SENS_DF)}</div>
    </div>
  </div>

  <div class="foot">
    ※ 본 리포트는 공개 OpenDART 시점고정(PIT) 데이터와 네이버 수정주가 기반의 전략 연구 자료입니다. 생존 편향(374개사 현존 기업) 한계가 존재하므로, 4중 관문을 통과한 경우에만 실전 스크리너 반영을 검토하시기 바랍니다.
  </div>
</div>
</body>
</html>"""

report_html = f"alpha_lab_report_{run_stamp}.html"
report_pdf = f"alpha_lab_report_{run_stamp}.pdf"
ic_csv = f"alpha_factor_ic_{run_stamp}.csv"
perf_csv = f"alpha_walkforward_{run_stamp}.csv"

Path(report_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, report_pdf, [("알파 팩터 연구실 대시보드", DASH_B64)])
ic_view.to_csv(ic_csv, index=False, encoding="utf-8-sig")
PERF_TABLE.to_csv(perf_csv, index=False, encoding="utf-8-sig")

print(f"[리포트 완료] PDF={'성공 (' + str(round(Path(report_pdf).stat().st_size/1024, 1)) + ' KB)' if pdf_ok and Path(report_pdf).exists() else '실패'} | HTML={Path(report_html).stat().st_size/1024:,.1f} KB")
if DRIVE_REPORT_DIR is not None:
    try:
        if pdf_ok and Path(report_pdf).exists():
            (DRIVE_REPORT_DIR / report_pdf).write_bytes(Path(report_pdf).read_bytes())
        (DRIVE_REPORT_DIR / report_html).write_text(html_report, encoding="utf-8")
        ic_view.to_csv(DRIVE_REPORT_DIR / ic_csv, index=False, encoding="utf-8-sig")
        PERF_TABLE.to_csv(DRIVE_REPORT_DIR / perf_csv, index=False, encoding="utf-8-sig")
        print(f"[Google Drive 자동 저장 완료] {DRIVE_REPORT_DIR / report_pdf}")
        print("  -> Google Drive 앱 -> Colab Notebooks/dividend_alpha_lab/reports/ 에서 PDF 바로 열기!")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")
# Google Drive 전용 저장 (브라우저 자동 다운로드 생략)
